In [2]:
TARGET_WORDS = ['yes', 'no', 'on', 'off', 'stop']
LABELS = ['silence', 'unknown'] + TARGET_WORDS

N_MFCC = 40
N_FRAMES = 101
N_INPUT = N_MFCC * N_FRAMES
N_OUTPUT = 7
FIXED_SCALE_IN = 1024.0
FIXED_SCALE_OUT = 4096.0

N_FFT = 400
HOP_LENGTH = 160
N_MELS = 40

INPUT_RATE = 48000
SAMPLE_RATE = 16000

STEP_SECONDS = 0.1
STEP_SAMPLES_RAW = int(INPUT_RATE * STEP_SECONDS)        
STEP_SAMPLES = int(SAMPLE_RATE * STEP_SECONDS)

WINDOW_SAMPLES = 16000         

running = False
COOLDOWN_PERIOD = 2.0
COMMAND_LOCKOUT = 1.5
last_command_time = 0
last_recognized_word = None

BLINK_INTERVAL = 0.02
RED_POS   = 0
BLUE_POS  = 2
is_blinking = False
color_index = 0
color_shifts = [RED_POS, BLUE_POS]
last_blink_time = 0

UNKNOWN_LOGIT_OFFSET = 1.4  
HISTORY_LEN = 3    
prediction_history = []






import sys
import numpy as np
import time

from pynq import Overlay, allocate
from scipy.signal import stft, resample_poly
from scipy.fftpack import dct

sys.path.append('/home/xilinx/jupyter_notebooks')

overlay = Overlay("final.bit")

pAudio = overlay.audio_codec_ctrl_0
dma = overlay.axi_dma_0
my_ip = overlay.myproject_0
leds = overlay.leds_gpio
rgb_gpio = overlay.rgbleds_gpio
btns = overlay.btns_gpio

pAudio.configure(sample_rate=48000, iic_index=1, uio_name="audio-codec-ctrl")
pAudio.select_microphone()

input_buffer = allocate(shape=(N_INPUT,), dtype=np.int32)
output_buffer = allocate(shape=(8,), dtype=np.int32)

input_buffer[:] = 0
output_buffer[:] = 0

audio_buffer = np.zeros(16000, dtype=np.float32)









def off_state():
    try:
        if hasattr(dma, 'sendchannel') and dma.sendchannel.running:
            dma.sendchannel.stop()
        if hasattr(dma, 'recvchannel') and dma.recvchannel.running:
            dma.recvchannel.stop()
            
        dma.sendchannel._mmio.write(0x00, 0x00000004)
        dma.recvchannel._mmio.write(0x30, 0x00000004)
    except Exception:
        print("er")
        pass
    
    overlay.leds_gpio.write(0x00, 0)
    rgb_gpio.write(0x00, 0)

def turn_off(): 
    global running, is_blinking, last_recognized_word, last_command_time
    
    leds.write(0x00, 0x0F)
    rgb_gpio.write(0x00, 1 << 1)
    time.sleep(1.0)
    
    running = False
    is_blinking = False
    last_recognized_word = None
    last_command_time = 0
    
    audio_buffer.fill(0)
    prediction_history.clear()
    
    off_state()
    
    
    
    
    
    

def get_samples(pAudio):
    pAudio.record(STEP_SECONDS)
    
    raw_buffer = np.asarray(pAudio.buffer, dtype=np.int32)
    
    if len(raw_buffer) < STEP_SAMPLES_RAW:
        return np.zeros(STEP_SAMPLES, dtype=np.float32)
        
    samples = raw_buffer[-STEP_SAMPLES_RAW:].copy()
    
    samples = samples.astype(np.int64)
    samples = samples & 0xFFFFFF
    samples[samples & 0x800000 != 0] -= 0x1000000
    
    samples_float = samples.astype(np.float32) / (2**23)
    samples_float = np.clip(samples_float, -1.0, 1.0)
    samples_float -= np.mean(samples_float)
    
    samples_16k = resample_poly(samples_float, 1, 3).astype(np.float32)
    
    if len(samples_16k) > STEP_SAMPLES:
        samples_16k = samples_16k[:STEP_SAMPLES]
    elif len(samples_16k) < STEP_SAMPLES:
        samples_16k = np.pad(samples_16k, (0, STEP_SAMPLES - len(samples_16k)), 'constant')
        
    return samples_16k


def create_mel_weight_matrix(num_mel_filter, num_spectrogram_coeffs, sample_rate, low_freq, high_freq):
    def hz_to_mel(hz):
        return 2595.0 * np.log10(1.0 + hz / 700.0)

    def mel_to_hz(mel):
        return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)
    
    low_mel = hz_to_mel(low_freq)
    higher_mel = hz_to_mel(high_freq)
    
    mel_frequencies = np.linspace(low_mel, higher_mel, num_mel_filter + 2)
    frequency_hz = mel_to_hz(mel_frequencies)

    frequency_ratio = (num_spectrogram_coeffs * 2) * frequency_hz / sample_rate
    frequency_idx = np.floor(frequency_ratio).astype(int)
    
    mel_filter_weights = np.zeros((num_spectrogram_coeffs, num_mel_filter))

    for i in range(num_mel_filter):
        start = frequency_idx[i]
        center = frequency_idx[i + 1]
        end = frequency_idx[i + 2]

        for j in range(start, center):
            if center != start:
                mel_filter_weights[j, i] = (j - start) / (center - start)

        for j in range(center, end):
            if end != center:
                mel_filter_weights[j, i] = (end - j) / (end - center)

    return mel_filter_weights

MEL_FILTER_WEIGHTS = create_mel_weight_matrix(
    num_mel_filter=N_MELS,
    num_spectrogram_coeffs=(N_FFT // 2 + 1),
    sample_rate=SAMPLE_RATE,
    low_freq=20.0,
    high_freq=4000.0
)

HANN_WINDOW = np.hanning(N_FFT)
HANN_SUM = np.sum(HANN_WINDOW)

def extract_mfcc(signal, sample_rate=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS):
    signal = np.asarray(signal, dtype=np.float32)
    padding = n_fft // 2
    padded_signal = np.pad(signal, (padding, padding), mode='reflect')

    f, t, Zxx = stft(
        padded_signal,
        fs=sample_rate,
        window='hann',
        nperseg=n_fft,
        noverlap=n_fft - hop_length,
        boundary=None,
        padded=False
    )
    
    amplitudes = np.abs(Zxx)
    spectrogram = amplitudes * HANN_SUM
    spectrogram = spectrogram.T
    
    mel_spectrogram = np.dot(spectrogram, MEL_FILTER_WEIGHTS)
    log_mel_spectrogram = np.log(mel_spectrogram + 1e-6)

    mfcc = dct(log_mel_spectrogram, type=2, axis=-1, norm='ortho')
    mfcc = mfcc[..., :n_mfcc]
  
    mfcc = mfcc.T
    mfcc = np.expand_dims(mfcc, axis=0)
    mfcc = np.expand_dims(mfcc, axis=-1)

    return mfcc.astype(np.float32)

def get_MFCC(chunk):
    if len(chunk) != STEP_SAMPLES:
        return False
        
    audio_buffer[:-STEP_SAMPLES] = audio_buffer[STEP_SAMPLES:]
    audio_buffer[-STEP_SAMPLES:] = chunk
    
    features = extract_mfcc(audio_buffer)
    
    mfcc_int32 = np.clip(np.round(features * FIXED_SCALE_IN), -2147483648, 2147483647).astype(np.int32)
    
    input_buffer[:] = mfcc_int32.flatten(order='C')
    output_buffer[:] = 0
    
    return True

In [5]:
import time
import numpy as np
import pandas as pd
from IPython.display import clear_output
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

TARGET_WORDS = ['yes', 'no', 'on', 'off', 'stop']

def run_single_test(duration_seconds, min_target_frames):
    global prediction_history
    prediction_history = []
    
    start_time = time.time()
    detected_labels = []
    
    count_by_words = {}
    valid_words = {}

    while (time.time() - start_time) < duration_seconds:
        chunk = get_samples(pAudio)
        if not get_MFCC(chunk):
            continue

        my_ip.write(0x00, 1)
        dma.recvchannel.transfer(output_buffer)
        dma.sendchannel.transfer(input_buffer)
        dma.sendchannel.wait()
        dma.recvchannel.wait()

        output_data = np.array(output_buffer, dtype=np.int32)
        model_output = np.zeros(7, dtype=np.float32)

        for i in range(7):
            bit_offset = i * 24
            word_idx = bit_offset // 32
            shift = bit_offset % 32
            val = (output_data[word_idx] >> shift) & 0xFFFFFF

            if shift > 8:
                bits1 = 32 - shift
                bits2 = 24 - bits1
                mask1 = (1 << bits1) - 1
                mask2 = (1 << bits2) - 1
                part1 = (output_data[word_idx] >> shift) & mask1
                part2 = output_data[word_idx + 1] & mask2
                val = part1 | (part2 << bits1)

            if val & 0x800000:
                val -= 0x1000000
            model_output[i] = val

        logits = model_output.astype(np.float32) / FIXED_SCALE_OUT
        logits[1] -= UNKNOWN_LOGIT_OFFSET
        logits_shifted = logits - np.max(logits)
        probabilities = np.exp(logits_shifted) / np.sum(np.exp(logits_shifted))

        prediction_history.append(probabilities)
        if len(prediction_history) > HISTORY_LEN:
            prediction_history.pop(0)
            
        probabilities = np.mean(prediction_history, axis=0)

        predicted_index = int(np.argmax(probabilities))
        confidence = float(probabilities[predicted_index])
        predicted_label = LABELS[predicted_index]

        detected_labels.append(predicted_index)

    if not detected_labels:
        return 'silence'

    for label in detected_labels:
        if label in count_by_words:
            count_by_words[label] += 1
        else:
            count_by_words[label] = 1

    for label_index, count in count_by_words.items():
        label = LABELS[label_index]

        if label in TARGET_WORDS and count >= min_target_frames:
            valid_words[label] = count

    if valid_words:
        final_pred = max(valid_words, key=valid_words.get)
    else:
        final_pred_index = max(count_by_words, key=count_by_words.get)
        final_pred = LABELS[final_pred_index]

    return final_pred


def start_eval(samples_per_class, countdown_seconds, between_class_pause):
    unknown_words = ['light', 'alarm', 'fan', 'ac', 'room', 'dog', 'cat', 'svjetlo', 'klima', 'start']

    total_classes = len(LABELS)

    unknown_prompts = []
    label_true = []
    label_prediction = []
    
    for word in unknown_words:
            for _ in range(2):
                unknown_prompts.append(word)

    for class_index, target in enumerate(LABELS, 1):
        if class_index > 1:
            for pause in range(between_class_pause, 0, -1):
                clear_output(wait=True)
                print(f"{target}")
                print(f"\n{pause}")
                time.sleep(1.0)

        if target == 'unknown':
            num_samples = len(unknown_prompts)
        else:
            num_samples = samples_per_class

        for i in range(num_samples):
            if target == 'silence':
                msg = "Riječ: silence"
                display_target = "silence"
            elif target == 'unknown':
                current_word = unknown_prompts[i]
                rep = (i % 2) + 1
                msg = f"Riječ: '{current_word}' ({rep}/2)"
                display_target = f"unknown: '{current_word}'"
            else:
                msg = f"{target}"
                display_target = target

            for countdown in range(countdown_seconds, 0, -1):
                clear_output(wait=True)
                print(f"Klasa {class_index}/{total_classes}: {target}")
                print(f"Ponavljanje: {i+1}/{num_samples}\n")

                if label_true:
                    last_true = label_true[-1]
                    last_pred = label_prediction[-1]
                    status_str = "True" if last_true == last_pred else "False"
                    
                    print(f"Prepoznata: {last_pred} - {status_str}\n")

                print(f"{msg}")
                print(f"Snimanje za: {countdown}")
                time.sleep(1.0)

            clear_output(wait=True)
            print(f"Klasa {class_index}/{total_classes}: {display_target}")
            print(f"Ponavljanje: {i+1}/{num_samples}")
            print("\nSnimanje")

            predicted = run_single_test(duration_seconds=2.0, min_target_frames=3)
            
            label_true.append(target)
            label_prediction.append(predicted)
            
            clear_output(wait=True)
            print(f"Stvarna: {target}\n")
            print(f"Prepoznata: {predicted}")
            time.sleep(1.2)

    clear_output(wait=True)
    
    matrica = confusion_matrix(label_true, label_prediction, labels=LABELS)

    rows = []
    columns = []
    for label in LABELS:
        rows.append(f"Točno: {label}")
        columns.append(f"Predviđeno: {label}")

    matrica_view = pd.DataFrame(matrica, index=rows, columns=columns)
    print("\n")
    print(matrica_view)
    
    report = classification_report(label_true, label_prediction, labels=LABELS, target_names=LABELS, digits=3, zero_division=0)
    print("\n")
    print(report)

    txt_filename = "rezultati_live_pynq.txt"
    with open(txt_filename, "w", encoding="utf-8") as f:
        f.write("Matrica zabune\n")
        f.write(matrica_view.to_string() + "\n\n")
        f.write("Metrike klasifikacije\n")
        f.write(report + "\n\n")
            
    img_filename = "matrica_zabune_pynq.png"
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matrica, interpolation='nearest', cmap=plt.cm.Blues)
    fig.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(matrica.shape[1]),
        yticks=np.arange(matrica.shape[0]),
        xticklabels=LABELS, 
        yticklabels=LABELS,
        ylabel='Stvarna klasa',
        xlabel='Predviđena klasa'
    )

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    thresh = matrica.max() / 2.
    for i in range(matrica.shape[0]):
        for j in range(matrica.shape[1]):
            ax.text(
                j, i, format(matrica[i, j], 'd'),
                ha="center", va="center",
                color="white" if matrica[i, j] > thresh else "black"
            )

    fig.tight_layout()
    
    plt.savefig(img_filename, dpi=300)
    plt.show()
    plt.close()

    return label_true, label_prediction, matrica_view

In [ ]:
label_true, label_prediction, matrica_view = start_eval(20, 3, 5)